In [21]:
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle 


In [22]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
#Preprocess the data 
data = data.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [24]:
## Encode the categorical variables 
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [25]:
# One-hot encode "Geography"
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df





,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [26]:
# Combine one-hot encoded columns with original data 
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [27]:
# Split the data into features and target 
x = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']


In [28]:
##Split the data in training and testing sets 
x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [29]:

##Scale these features 
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [30]:
# Save the encoders and scaler for later use 
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)    

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)   

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)



In [31]:
##### Train our ANN Regression problem statement 
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [33]:
## Build Our ANN model 
model = Sequential([
    Dense(64, activation='relu',input_shape=(x_train.shape[1],)), ## this is first hidden layer HL1, connected with input layer 
    Dense(32, activation='relu'), ## HIDDEN LAYER 2, HL2 
    Dense(1) ## HL3, output layer

])

## compile the model 
model.compile(optimizer='adam', loss='mean_squared_error',metrics=['mae'])

model.summary()



/Users/nidhikumari/Desktop/ANN_CLASSIFICATION_APP_UDEMY_KRISHNAIK/venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#  set logs to train yhe model with early stopping and tensorboard callbacks
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard 
import datetime

#Set up TensorBoard 
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [ ]:
#Set up Early Stopping , we use validation loss to monitor the training and we set patience to 10, which means if the validation loss does not improve for 10 consecutive epochs, the training will be stopped and the best weights will be restored.
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)



In [39]:
# train the model 
history = model.fit(
    x_train, y_train, 
    validation_data=(x_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)



Epoch 1/100


2026-05-16 18:29:09.490063: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 7529690624.0000 - mae: 70882.1094 - val_loss: 6913796096.0000 - val_mae: 68109.4297
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 7095770624.0000 - mae: 68690.0859 - val_loss: 6639888896.0000 - val_mae: 66636.7266
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 6537400832.0000 - mae: 65895.9219 - val_loss: 5758889984.0000 - val_mae: 62146.6328
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 5196664832.0000 - mae: 59322.8750 - val_loss: 4288861184.0000 - val_mae: 54877.2266
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 3918202624.0000 - mae: 52843.3203 - val_loss: 3600532224.0000 - val_mae: 51348.2422
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 3561359104.0000 - mae: 51014.9531 - val_loss: 3488090368.0000 - val_mae: 50704.8906
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 3512012032.0000 - mae: 50785.1484 - val_loss: 3490879232.0000 - val_mae: 50681.99

In [40]:
%load_ext tensorboard 


In [42]:
#Launching the tensorboard session 
%tensorboard --logdir regressionlogs/fit 


Reusing TensorBoard on port 6007 (pid 32976), started 0:00:40 ago. (Use '!kill 32976' to kill it.)

In [43]:
## Evaluate the model on the test set
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f'Test MAE : {test_mae}')
print(f'Test Loss : {test_loss}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3487235584.0000 - mae: 50668.6367
Test MAE : 50668.63671875
Test Loss : 3487235584.0


In [44]:
## save the model 
model.save('regression_model.h5')